
Analytic question: Do Forwards record a significantly higher average of Shots on Target per 90 minutes than Midfielders?

Data Wrangling
This notebook performs data wrangling on the raw player dataset. It selects relevant variables, checks and handles data-quality issues, applies the participation and position inclusion criteria, restructures position information, assigns unique player IDs, and exports the analysis-ready population for the next stage.


In [1]:


import pandas as pd  #import pandas for data manipulation
import os #import os for file path operations

# Raw input file supplied for the assignment.
csv_path = "Player_Shots_on_Target_data.csv"

if not os.path.exists(csv_path): #Check if the CSV file exists in the current directory
    raise FileNotFoundError(    #Raise an error if the file is not found
        "Place 'Player_Shots_on_Target_data.csv' in the same folder as this notebook, then run again." #provide instructions to the user if the file is not found to put the data file in the same folder as the notebook and run again
    )

df = pd.read_csv(csv_path, encoding="cp1252") #Read the CSV file into a pandas DataFrame with specified encoding

print("Raw dataset shape:", df.shape) #Print the shape of the raw dataset
display(df.head()) #Dispay the first few rows of the DataFrame to give an overview of the data


Raw dataset shape: (1039, 18)


,Rk,Player,Pos,Squad,Age,Born,90s,Gls,Sh,SoT,SoT%,Sh/90,SoT/90,G/Sh,G/SoT,PK,PKatt,Matches
0,1,Brenden Aaronson,MF,us USA,25,2000,0.8,0,3,1,33.3,3.55,1.18,0.0,0.0,0,0,Matches
1,2,Thelo Aasgaard,MF,no Norway,24,2002,1.0,1,2,1,50.0,2.00,1.00,0.5,1.0,0,0,Matches
2,3,Hamza Abdelkarim,FW,eg Egypt,18,2008,0.7,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,Matches
3,4,Hossam Abdelmaguid,DF,eg Egypt,25,2001,0.7,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,Matches
4,5,Mohamed Abdelmonem,DF,eg Egypt,27,1999,0.2,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,Matches


In [2]:
# Select only variables relevant to the analytic question.
required_columns = ["Player", "Pos", "Squad", "Age", "90s", "Sh", "SoT", "SoT/90"]

missing_columns = [col for col in required_columns if col not in df.columns]  #Check for missing required columns in the DataFrame and raise an error if any are missing
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_clean = df[required_columns].copy() #create a copy of the DataFrame with only the required columns for further analysis so that the original DataFrame remains unchanged

# Check missing values before cleaning.
print("Missing values before cleaning:")
display(df_clean.isna().sum().to_frame("Missing_Count"))


Missing values before cleaning:


,Missing_Count
Player,0
Pos,0
Squad,0
Age,0
90s,0
Sh,0
SoT,0
SoT/90,0


In [3]:
# Remove duplicate rows if any are present.
duplicates_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy() #create a copy of the DataFrame after dropping duplicate rows to ensure the original DataFrame remains unchanged

# Keep players who actually appeared (more than 0 equivalent 90-minute periods).
df_clean = df_clean[df_clean["90s"] > 0].copy()

# Use the first listed position as the player's primary position.
df_clean["Primary_Position"] = df_clean["Pos"].str.split(",").str[0].str.strip()

# Keep only Forwards and Midfielders.
df_clean = df_clean[df_clean["Primary_Position"].isin(["FW", "MF"])].copy()  #This is in case the player's position is a combination of either FW or MF with any other position, we will keep only those players whose primary position is either FW or MF.

# Give the groups clear labels.
df_clean["Position_Group"] = df_clean["Primary_Position"].map(
    {"FW": "Forward", "MF": "Midfielder"}
)

# Remove records missing the outcome or position group.
df_clean = df_clean.dropna(subset=["SoT/90", "Position_Group"]).copy()

# Assign a reproducible unique ID after cleaning.
df_clean = df_clean.reset_index(drop=True) #first reset the index of the DataFrame to ensure that the new Player_IDs are assigned in a sequential manner starting from 1, and drop the old index to avoid confusion.
df_clean["Player_ID"] = ["PL" + str(i).zfill(4) for i in range(1, len(df_clean) + 1)]

# Display the results of the cleaning process.
print("Duplicate rows removed:", duplicates_before)
print("Wrangled dataset shape:", df_clean.shape)
print("\nEligible players by position:")
print(df_clean["Position_Group"].value_counts())
display(df_clean.head())


Duplicate rows removed: 0
Wrangled dataset shape: (608, 11)

Eligible players by position:
Position_Group
Midfielder    385
Forward       223
Name: count, dtype: int64


,Player,Pos,Squad,Age,90s,Sh,SoT,SoT/90,Primary_Position,Position_Group,Player_ID
0,Brenden Aaronson,MF,us USA,25,0.8,3,1,1.18,MF,Midfielder,PL0001
1,Thelo Aasgaard,MF,no Norway,24,1.0,2,1,1.00,MF,Midfielder,PL0002
2,Hamza Abdelkarim,FW,eg Egypt,18,0.7,0,0,0.00,FW,Forward,PL0003
3,Yusuf Abdurisag,FW,qa Qatar,26,1.1,0,0,0.00,FW,Forward,PL0004
4,Elias Achouri,"FW,MF",tn Tunisia,27,0.5,3,1,2.09,FW,Forward,PL0005


In [4]:
# Final quality checks.
print("Missing values after cleaning:")
display(df_clean.isna().sum().to_frame("Missing_Count")) #Check for missing values after cleaning and display the count of missing values for each column in a DataFrame format

print("\nSoT/90 data type:", df_clean["SoT/90"].dtype)
print("Unique Player_ID values:", df_clean["Player_ID"].nunique())
print("Total wrangled rows:", len(df_clean))


Missing values after cleaning:


,Missing_Count
Player,0
Pos,0
Squad,0
Age,0
90s,0
Sh,0
SoT,0
SoT/90,0
Primary_Position,0
Position_Group,0



SoT/90 data type: float64
Unique Player_ID values: 608
Total wrangled rows: 608


In [5]:
# Export the wrangled population for Notebook 2 and later population checks.
output_file = "Wrangled_player_data.csv"
df_clean.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Exported: {output_file}")


Exported: Wrangled_player_data.csv
